# 1. Logistic Regression — Bank Marketing Classification

This notebook implements Logistic Regression to predict whether a customer will subscribe to a term deposit.

The model uses the same Bank Marketing dataset and the reusable preprocessing pipeline developed earlier.

The goal is to train, tune, and evaluate a Logistic Regression classifier using stratified data splits.

It reuses the project's preprocessing and Logistic Regression modules for consistent training and evaluation.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Project constants
TARGET_COLUMN = "y"
RANDOM_STATE = 42

# Identify project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import reusable preprocessing function
from src.preprocessing import build_preprocessor

# Added to reuse the LR
from src.logistic_regression import (
    build_logistic_pipeline,
    tune_logistic_regression
)

print("Logistic Regression environment ready.")

Logistic Regression environment ready.


## 2. Data Loading and Splitting

We use the Bank Marketing dataset to predict whether a customer will subscribe to a term deposit (`y`).

The target variable is converted to binary values (1 = Yes, 0 = No). We exclude selected features based on our preprocessing decisions and split the data into training (70%), validation (15%), and testing (15%) sets.

Stratified sampling preserves the class distribution across all three datasets.

In [2]:
# Load dataset
df = pd.read_csv(PROJECT_ROOT / "data" / "bank-full.csv", sep=";")

# Encode target variable
y = df[TARGET_COLUMN].map({"yes": 1, "no": 0})

# Exclude selected features (same as preprocessing notebook)
features_to_exclude = ["default", "day", "duration", "pdays", "poutcome"]

X = df.drop(columns=[TARGET_COLUMN] + features_to_exclude)

# Identify feature types
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

# Stratified 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# Verify dataset dimensions
print("Original dataset:", df.shape)
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("\nNumerical features:", numerical_features)
print("Categorical features:", categorical_features)

# Verify target distribution
for name, target in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Testing", y_test)
]:
    print(f"{name} positive class: {target.mean() * 100:.2f}%")

Original dataset: (45211, 17)
Training: (31647, 11)
Validation: (6782, 11)
Testing: (6782, 11)

Numerical features: ['age', 'balance', 'campaign', 'previous']
Categorical features: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'month']
Training positive class: 11.70%
Validation positive class: 11.69%
Testing positive class: 11.71%


## 3. Build Logistic Regression Pipeline

We created a machine learning pipeline that combines numerical and categorical preprocessing with Logistic Regression.

The pipeline ensures that preprocessing is fitted only on the training data, preventing data leakage.

In [3]:
# Create Logistic Regression pipeline
logistic_pipeline = build_logistic_pipeline(
    numerical_features,
    categorical_features,
    random_state=RANDOM_STATE
)

print("Logistic Regression pipeline created successfully.")
print(logistic_pipeline)

Logistic Regression pipeline created successfully.
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'balance', 'campaign',
                                                   'previous']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                   

## 4. Train Baseline Logistic Regression Model

We trained the Logistic Regression pipeline using the training dataset.

The pipeline automatically applies preprocessing before fitting the classifier. The validation and test datasets are not used during training.

In [4]:
# Train baseline Logistic Regression model
logistic_pipeline.fit(X_train, y_train)

print("Baseline Logistic Regression model trained successfully.")

# Verify fitted model
print("\nModel classes:", logistic_pipeline.classes_)
print("Number of features:",
      logistic_pipeline.named_steps["classifier"].n_features_in_)

print("Iterations used:",
      logistic_pipeline.named_steps["classifier"].n_iter_)

Baseline Logistic Regression model trained successfully.

Model classes: [0 1]
Number of features: 42
Iterations used: [32]


## 5. Evaluate Baseline Logistic Regression Model

We evaluated the baseline Logistic Regression model on the validation dataset using accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix to measure classification performance.

In [5]:
# Predict on validation dataset
y_val_pred = logistic_pipeline.predict(X_val)
y_val_prob = logistic_pipeline.predict_proba(X_val)[:, 1]

# Calculate evaluation metrics
baseline_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred),
    "F1-score": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

# Display results
print("Baseline Logistic Regression — Validation Results")
display(pd.DataFrame([baseline_results]).round(4))

# Display confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

Baseline Logistic Regression — Validation Results


,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,0.8825,0.4853,0.0832,0.1421,0.754


Confusion Matrix:
[[5919   70]
 [ 727   66]]


## 6. Hyperparameter Tuning

We use GridSearchCV with stratified 5-fold cross-validation to find the best Logistic Regression hyperparameters.

The search evaluates different regularization strengths (`C`) and class weighting options (`class_weight`).

We use F1-score as the evaluation metric because the dataset is imbalanced, and we want to balance precision and recall for the minority class (`Yes`).

In [6]:
# Tune Logistic Regression using stratified cross-validation
grid_search = tune_logistic_regression(
    logistic_pipeline,
    X_train,
    y_train,
    random_state=RANDOM_STATE
)

# Display tuning results
print("Best Parameters:", grid_search.best_params_)
print(f"Best Cross-Validation F1-score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters: {'classifier__C': 1, 'classifier__class_weight': 'balanced'}
Best Cross-Validation F1-score: 0.3427


## 7. Baseline vs. Tuned Logistic Regression

We evaluate the tuned model on the validation dataset and compare its performance with the baseline model.

The comparison focuses on accuracy, precision, recall, F1-score, and ROC-AUC to understand how hyperparameter tuning affects classification performance.

In [7]:
# Get best tuned model
best_logistic_model = grid_search.best_estimator_

# Predict on validation dataset
y_val_pred_tuned = best_logistic_model.predict(X_val)
y_val_prob_tuned = best_logistic_model.predict_proba(X_val)[:, 1]

# Calculate evaluation metrics
tuned_results = {
    "Accuracy": accuracy_score(y_val, y_val_pred_tuned),
    "Precision": precision_score(y_val, y_val_pred_tuned, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred_tuned, zero_division=0),
    "F1-score": f1_score(y_val, y_val_pred_tuned, zero_division=0),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob_tuned)
}

# Compare baseline and tuned models
comparison = pd.DataFrame(
    [baseline_results, tuned_results],
    index=["Baseline", "Tuned"]
)

print("Baseline vs Tuned Logistic Regression")
display(comparison.round(4))

# Confusion matrix
print("\nTuned Model Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_tuned))


Baseline vs Tuned Logistic Regression


,Accuracy,Precision,Recall,F1-score,ROC-AUC
Baseline,0.8825,0.4853,0.0832,0.1421,0.7540
Tuned,0.6995,0.2309,0.6734,0.3439,0.7533



Tuned Model Confusion Matrix:
[[4210 1779]
 [ 259  534]]


### Results Interpretation

The tuned Logistic Regression model significantly improved recall from 8.32% to 67.34% and F1-score from 0.1421 to 0.3439.

However, accuracy decreased from 88.25% to 69.95%, and precision also decreased. This happened because the balanced class weights encouraged the model to identify more potential subscribers, resulting in more false positives.

ROC-AUC remained nearly unchanged, suggesting that tuning primarily changed the classification behavior rather than improving the model's ability to distinguish between the two classes.

Overall, the tuned model performs better for our F1-focused objective because it identifies substantially more actual subscribers.

## 8. Final Test Evaluation

We evaluate the tuned Logistic Regression model on the unseen test dataset to measure its generalization performance.

We report accuracy, precision, recall, F1-score, ROC-AUC, and the confusion matrix.

The classification report provides additional information about model performance for subscribers (`Yes`) and non-subscribers (`No`).

In [8]:
# Final evaluation on untouched test dataset
y_test_pred = best_logistic_model.predict(X_test)
y_test_prob = best_logistic_model.predict_proba(X_test)[:, 1]

# Calculate test metrics
test_results = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_test_prob)
}

# Display evaluation metrics
print("Tuned Logistic Regression — Test Results")
display(pd.DataFrame([test_results]).round(4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Tuned Logistic Regression — Test Results


,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,0.7014,0.235,0.6877,0.3503,0.7594



Confusion Matrix:
[[4211 1777]
 [ 248  546]]


### Test Results Interpretation

The tuned Logistic Regression model achieved an accuracy of 70.14%, recall of 68.77%, F1-score of 0.3503, and ROC-AUC of 0.7594.

The model correctly identified 546 out of 794 actual subscribers but missed 248 subscribers. It also incorrectly classified 1,777 non-subscribers as subscribers.

Although the model produces many false positives, its relatively high recall makes it more effective at identifying potential subscribers than the baseline model.

The test results are close to the validation results, suggesting that the model generalizes reasonably well to unseen data.

## 9. Save and Verify the Trained Model

The tuned Logistic Regression pipeline is saved using `joblib` for future predictions.

The saved pipeline includes preprocessing and the trained classifier, allowing new data to be processed and classified consistently without retraining.

In [9]:
# Create models directory if it does not exist
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save trained Logistic Regression pipeline
model_path = models_dir / "logistic_regression.joblib"

joblib.dump(best_logistic_model, model_path)

print("Logistic Regression model saved successfully.")

# Verify saved model
loaded_model = joblib.load(model_path)

# Confirm loaded model produces identical predictions
original_predictions = best_logistic_model.predict(X_test)
loaded_predictions = loaded_model.predict(X_test)

print("Predictions match:",
      np.array_equal(original_predictions, loaded_predictions))

Logistic Regression model saved successfully.
Predictions match: True
